# Day 2 — From tools to a working agent

This notebook takes **`shared_tools.py`** and **`agent.py`** apart, one piece at a time,
and runs each piece on its own so you can see what it actually produces.

Day 1 ended with a RAG pipeline: a PDF turned into a searchable vector store. Day 2 turns
that into something that *acts*.

```
Part A   shared_tools.py   ->  three TOOLS the model is allowed to call
Part B   agent.py          ->  a MODEL that decides which to call, behind a web UI
```

**Part A — the tools**

| Tool | What it does | Where the data comes from |
|---|---|---|
| `web_search_tool` | Search the live internet | DuckDuckGo (ready-made tool) |
| `get_job_recommendation` | Find Indian job listings | Adzuna REST API |
| `get_resume_data` | Look things up in the resume | Local Chroma vector store (Day 1's RAG) |

1. Setup and environment
2. A ready-made tool — `DuckDuckGoSearchRun`
3. A plain Python function — *before* the decorator
4. The same function as a tool — *after* `@tool`
5. What the LLM actually sees — tool introspection
6. A RAG-backed tool — `get_resume_data`
7. The finished toolbox

**Part B — the agent**

8. The model — `llm.py`
9. The system prompt — `system_prompts.py`
10. Building and running the agent — `create_agent`
11. Moving the tools out — `mcp_server.py` and MCP
12. The UI — Gradio

---

### Before you start

```bash
# 1. environment
uv venv --python 3.12 .venv          # or: python3.12 -m venv .venv
source .venv/bin/activate
uv pip install -r requirements.txt   # or: pip install -r requirements.txt

# 2. the local model (needed from §8 onward)
ollama pull qwen3:0.6b
ollama serve

# 3. API keys (needed for §3, §4)
cp .env.example .env                 # then paste your Adzuna keys in
```

**Every cell is written to fail politely.** If Ollama is not running, or the MCP server
is down, or your Adzuna keys are missing, the cell prints what to start instead of
throwing a traceback — so you can read the whole notebook top to bottom regardless, then
come back and re-run the parts you have set up.

---
## 1. Setup and environment

`shared_tools.py` opens with its imports and `load_dotenv()`.

```python
import os
from dotenv import load_dotenv
import requests

from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

from rag import vector_store

load_dotenv()
```

`load_dotenv()` reads the `.env` file in this folder and puts each line into the
environment, so `os.getenv("ADZUNA_APP_ID")` can find it later. Secrets stay out of
the code; `.env` is in `.gitignore`.

In [1]:
import os
import json
import requests
from dotenv import load_dotenv

from langchain_core.tools import tool

load_dotenv()

print("Python packages loaded.\n")
print("Environment check:")
for key in ["ADZUNA_APP_ID", "ADZUNA_API_KEY"]:
    value = os.getenv(key)
    if value:
        print(f"  {key:<16} set  (ends with ...{value[-4:]})")
    else:
        print(f"  {key:<16} MISSING  -> add it to your .env file")

Python packages loaded.

Environment check:
  ADZUNA_APP_ID    MISSING  -> add it to your .env file
  ADZUNA_API_KEY   MISSING  -> add it to your .env file


---
## 2. A ready-made tool — `DuckDuckGoSearchRun`

```python
web_search_tool = DuckDuckGoSearchRun()
```

One line. LangChain ships dozens of pre-built tools, and this is one of them — you do
not write the search logic yourself.

Two things to notice:

* It is an **object**, not a function. You run it with **`.invoke(...)`**, not `(...)`.
* It returns **plain text**, not JSON. Whatever comes back is dropped straight into the
  LLM's context, so the text itself *is* the answer the model reads.

In [2]:
from langchain_community.tools import DuckDuckGoSearchRun

web_search_tool = DuckDuckGoSearchRun()

print("Type   :", type(web_search_tool).__name__)
print("Name   :", web_search_tool.name)
print("Descr. :", web_search_tool.description)

Type   : DuckDuckGoSearchRun
Name   : duckduckgo_search
Descr. : A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.


/var/folders/kg/3w3p831j3ks72cgsrc3crrfr0000gq/T/ipykernel_72372/1406052059.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [3]:
# The commented-out line from shared_tools.py:
#     response = web_search_tool.invoke("LA Olympics 2028")

try:
    response = web_search_tool.invoke("LA Olympics 2028")
    print(response[:1500])
except Exception as e:
    print("Search failed (rate limit or no network):", type(e).__name__)
    print(e)

The 2028 Summer Olympics, officially the Games of the XXXIV Olympiad and commonly known as Los Angeles 2028 or LA28, is an upcoming international multi-sport event scheduled to take place from July 14 to 30, 2028, in the United States. Official Olympic coverage of LA 2028. Read the latest stories from the Olympic Movement across the globe. The Olympic Games Los Angeles 2028 will run from 14-30 July 2028, staging 51 sports disciplines across more than 40 venues stretching from downtown LA to as far afield as Oklahoma City and New York. No new permanent structures will be built for the occasion - a first for any Summer Games since London 1948 - with organisers instead drawing on Southern California's deep roster of world ... See when and where the world's best will compete in Los Angeles! Full artistic, trampoline, and rhythmic gymnastics schedule for the 2028 Olympics — all dates, times (PT), events, and venues. Plan your dream trip to the LA 2028 Olympics with this ultimate guide—cover

> **Note the failure mode.** DuckDuckGo rate-limits aggressively. If the cell above
> errors, that is normal — wait a few seconds and re-run. This matters for agents:
> a tool that fails intermittently is something the agent has to cope with at runtime.

---
## 3. A plain Python function — *before* the decorator

Before we make `get_job_recommendation` a tool, let's look at it as an ordinary function.
It does one thing: build an Adzuna URL and `GET` it.

```python
def get_job_recommendation(what: str, salary_min: int) -> str:
    url = f'https://api.adzuna.com/v1/api/jobs/in/search/1?app_id=...&what={what}&salary_min={salary_min}'
    response = requests.get(url)
    return response.json()
```

Nothing AI-specific here — it is a REST call.

In [4]:
def get_job_recommendation_plain(what: str, salary_min: int) -> str:
    """Plain version - no @tool decorator."""
    url = (
        f'https://api.adzuna.com/v1/api/jobs/in/search/1'
        f'?app_id={os.getenv("ADZUNA_APP_ID")}'
        f'&app_key={os.getenv("ADZUNA_API_KEY")}'
        f'&what={what}'
        f'&salary_min={salary_min}'
    )
    response = requests.get(url)
    return response.json()


# A plain function is called the normal way: with parentheses.
result = get_job_recommendation_plain("Data Analyst", 300000)

print("Type returned:", type(result).__name__)
print(json.dumps(result, indent=2)[:1200])

Type returned: dict
{
  "display": "Authorisation failed",
  "__CLASS__": "Adzuna::API::Response::Exception",
  "exception": "AUTH_FAIL",
  "doc": "https://api.adzuna.com/v1/doc"
}


If you see an **authorisation error** above, your `.env` has no Adzuna credentials yet.
Get free keys at <https://developer.adzuna.com/> and put them in `.env`:

```
ADZUNA_APP_ID=your_app_id
ADZUNA_API_KEY=your_api_key
```

The rest of the notebook still works without them.

---
## 4. The same function as a tool — *after* `@tool`

Now add one line: `@tool`.

```python
@tool
def get_job_recommendation(what: str, salary_min: int) -> str:
    """..."""
```

`@tool` converts the function into a **`StructuredTool`** object. That object carries
everything the LLM needs in order to decide to call it:

* the **name** — taken from the function name
* the **arguments and their types** — taken from the type hints
* the **description** — taken from the **docstring**

The docstring stops being a note for humans. It becomes the instruction the model reads.
That is why the real docstring in `shared_tools.py` is so long and so bossy — *"This is
NOT a job title phrase"*, *"Not USD"*. Every one of those lines is there because the
model got it wrong without it.

In [5]:
@tool
def get_job_recommendation(what: str, salary_min: int) -> str:
    """
    Searches live Indian job listings (Adzuna) by keyword and minimum salary.

    Input Parameters:
    what (str) - One or more SINGLE KEYWORDS, space-separated. This is NOT a job
        title phrase - each space-separated word is matched as a separate keyword,
        not as part of one title. Use 1-2 core keywords, not a full title.
        Good: "python", "datascientist", "python fastapi", "machinelearning".
        Avoid: "machine learning engineer" (three separate required keywords,
        which over-narrows results since a listing must contain all three).
        Never pass a parameter name (like "salary_min") or an instruction as this value.
    salary_min (int) - Minimum ANNUAL salary in Indian Rupees (INR), e.g. 400000 for 4 LPA.
        Not USD. Typical entry-level India tech salaries start around 300000-600000.

    Returns:
    JSON with a "results" list. Each job object includes: title, company.display_name,
    description, salary_min, salary_max (all INR), location.display_name, and
    redirect_url (the direct apply link for that job).

    Example: get_job_recommendation(what="python", salary_min=400000)
    Example: get_job_recommendation(what="datascientist", salary_min=500000)
    """
    url = (
        f'https://api.adzuna.com/v1/api/jobs/in/search/1'
        f'?app_id={os.getenv("ADZUNA_APP_ID")}'
        f'&app_key={os.getenv("ADZUNA_API_KEY")}'
        f'&what={what}'
        f'&salary_min={salary_min}'
    )
    response = requests.get(url)
    return response.json()


print("It is no longer a function:", type(get_job_recommendation))

It is no longer a function: <class 'langchain_core.tools.structured.StructuredTool'>


### 4a. The old way of calling it now breaks

This is the demo from the bottom of `shared_tools.py`:

```python
print(get_job_recommendation("Data Analyst", 300000))   # worked BEFORE the decorator
```

Run it against the decorated version and it fails — the object is a tool now, not a
function.

In [6]:
try:
    print(get_job_recommendation("Data Analyst", 300000))
except Exception as e:
    print("FAILS as expected")
    print(f"  {type(e).__name__}: {e}")

FAILS as expected
  TypeError: 'StructuredTool' object is not callable


### 4b. The right way: `.invoke()` with a dictionary

Arguments go in as a **dict**, keyed by parameter name — exactly the shape the LLM will
produce when it decides to call this tool.

In [7]:
result = get_job_recommendation.invoke({"what": "Data Analyst", "salary_min": 300000})

print("Type returned:", type(result).__name__)
if isinstance(result, dict) and "results" in result:
    print(f"Jobs found: {len(result['results'])}\n")
    for job in result["results"][:3]:
        print("-", job.get("title"))
        print("  company :", job.get("company", {}).get("display_name"))
        print("  location:", job.get("location", {}).get("display_name"))
        print("  salary  :", job.get("salary_min"), "-", job.get("salary_max"))
        print("  apply   :", job.get("redirect_url", "")[:80])
        print()
else:
    print(json.dumps(result, indent=2)[:800])

Type returned: dict
{
  "doc": "https://api.adzuna.com/v1/doc",
  "display": "Authorisation failed",
  "exception": "AUTH_FAIL",
  "__CLASS__": "Adzuna::API::Response::Exception"
}


---
## 5. What the LLM actually sees

```python
print(f"\nTool Name: {get_job_recommendation.name}")
print(f"\nTool Arguments: {get_job_recommendation.args}")
print(f"\nTool Description: {get_job_recommendation.description}")
```

This is the single most useful cell in the notebook. When you bind tools to a model,
*this* — and nothing else — is what gets sent along with the prompt. The model never sees
your function body. It picks a tool purely from the name, the argument schema, and the
description.

**If the agent calls the wrong tool, or calls it with junk arguments, the fix is almost
always in the docstring.**

In [8]:
print("TOOL NAME")
print(" ", get_job_recommendation.name)

print("\nTOOL ARGUMENTS")
print(json.dumps(get_job_recommendation.args, indent=2))

print("\nTOOL DESCRIPTION  (this is the docstring, verbatim)")
print(get_job_recommendation.description)

TOOL NAME
  get_job_recommendation

TOOL ARGUMENTS
{
  "what": {
    "title": "What",
    "type": "string"
  },
  "salary_min": {
    "title": "Salary Min",
    "type": "integer"
  }
}

TOOL DESCRIPTION  (this is the docstring, verbatim)
Searches live Indian job listings (Adzuna) by keyword and minimum salary.

Input Parameters:
what (str) - One or more SINGLE KEYWORDS, space-separated. This is NOT a job
    title phrase - each space-separated word is matched as a separate keyword,
    not as part of one title. Use 1-2 core keywords, not a full title.
    Good: "python", "datascientist", "python fastapi", "machinelearning".
    Avoid: "machine learning engineer" (three separate required keywords,
    which over-narrows results since a listing must contain all three).
    Never pass a parameter name (like "salary_min") or an instruction as this value.
salary_min (int) - Minimum ANNUAL salary in Indian Rupees (INR), e.g. 400000 for 4 LPA.
    Not USD. Typical entry-level India tech salar

In [9]:
# The exact JSON schema shipped to the model:
print(json.dumps(get_job_recommendation.args_schema.model_json_schema(), indent=2))

{
  "description": "Searches live Indian job listings (Adzuna) by keyword and minimum salary.\n\nInput Parameters:\nwhat (str) - One or more SINGLE KEYWORDS, space-separated. This is NOT a job\n    title phrase - each space-separated word is matched as a separate keyword,\n    not as part of one title. Use 1-2 core keywords, not a full title.\n    Good: \"python\", \"datascientist\", \"python fastapi\", \"machinelearning\".\n    Avoid: \"machine learning engineer\" (three separate required keywords,\n    which over-narrows results since a listing must contain all three).\n    Never pass a parameter name (like \"salary_min\") or an instruction as this value.\nsalary_min (int) - Minimum ANNUAL salary in Indian Rupees (INR), e.g. 400000 for 4 LPA.\n    Not USD. Typical entry-level India tech salaries start around 300000-600000.\n\nReturns:\nJSON with a \"results\" list. Each job object includes: title, company.display_name,\ndescription, salary_min, salary_max (all INR), location.display_

---
## 6. A RAG-backed tool — `get_resume_data`

The third tool does not call an external API. It queries the **local vector store** that
`rag.py` built out of `resume.pdf`.

```python
from rag import vector_store

@tool
def get_resume_data(query: str) -> str:
    """..."""
    response = vector_store.similarity_search(query)
    return response
```

The pipeline underneath it, from `rag.py`:

```
resume.pdf
   -> PyPDFLoader                     (PDF  -> text)
   -> RecursiveCharacterTextSplitter  (text -> 500-char chunks, 150 overlap)
   -> HuggingFaceEmbeddings           (chunk -> 384-dim vector)
   -> Chroma (./vector_db)            (stored on disk)
```

`similarity_search` embeds the *query* the same way, then returns the chunks whose
vectors sit closest to it.

### 6a. Build the vector store

The first run downloads the embedding model (~90 MB) and creates `./vector_db/`.
After that it loads from disk.

In [10]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# --- chunking (rag.py: process_pdf) ---
loader = PyPDFLoader("./resume.pdf")
document = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=150)
chunks = splitter.split_documents(document)

print(f"Pages in PDF : {len(document)}")
print(f"Chunks made  : {len(chunks)}")
print("\n--- first chunk ---")
print(chunks[0].page_content)

Pages in PDF : 2
Chunks made  : 12

--- first chunk ---
ARJUN PRAKASH MENON
Junior AI Engineer
Bengaluru, Karnataka, India | +91 98450 22671 | arjun.p.menon2003@gmail.com | linkedin.com/in/arjunpmenon | github.com/arjunpmenon
PROFESSIONAL SUMMARY
Junior AI Engineer with hands-on experience building and deploying machine learning and NLP models through internships and 
academic projects. Proficient in Python, PyTorch, and scikit-learn, with practical exposure to LLM fine-tuning, RAG pipelines, and REST


In [11]:
# --- embed + store (rag.py: embeddings / vector_store / ingest_data) ---
# HuggingFaceEmbeddings needs the `sentence-transformers` package (it pulls in torch,
# ~2 GB). If it is missing, everything below still reads correctly - it just will not
# produce live output until you run:  uv pip install sentence-transformers
vector_store = None
try:
    from langchain_huggingface import HuggingFaceEmbeddings
    from langchain_chroma import Chroma

    embeddings = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")

    vector_store = Chroma(
        embedding_function=embeddings,
        collection_name="resume_data_collection",
        persist_directory="./vector_db",
    )

    # Only ingest if the collection is empty, so re-running does not duplicate chunks.
    if vector_store._collection.count() == 0:
        vector_store.add_documents(chunks)
        print("Data ingestion completed.")
    else:
        print("Collection already populated - skipping ingestion.")

    print("Chunks in store:", vector_store._collection.count())

except ImportError as e:
    print("Embedding model not installed - section 6 will not run live.")
    print(f"  {type(e).__name__}: {e}\n")
    print("To enable it:   uv pip install sentence-transformers")

Embedding model not installed - section 6 will not run live.
  ImportError: Could not import sentence_transformers python package. Please install it with `pip install sentence-transformers`.

To enable it:   uv pip install sentence-transformers


### 6b. The tool itself

Same pattern as before: a docstring that tells the model what a *good* query looks like.
Note the warning in it — ask for a **topic that appears in the resume** ("skills", "work
experience"), not for a judgement ("seniority level"). Similarity search matches text
against text; it cannot reason.

In [12]:
@tool
def get_resume_data(query: str) -> str:
    """
    Retrieves chunks of the candidate's resume via similarity search.

    Input Parameters:
    query (str) - A resume TOPIC to search for, e.g. "skills", "work experience",
        "education", "years of experience". This must be a topic that appears IN
        the resume itself - not an instruction, judgment, or classification task
        (e.g. do NOT query "seniority level classification"; instead query "work
        experience" and determine seniority yourself from what's returned).

    Returns:
    A list of matching resume text chunks. Content is chunked, so multiple calls
    with different topic queries are usually needed to get a full picture.

    Example: get_resume_data(query="skills")
    """
    response = vector_store.similarity_search(query)
    return response


# The line that runs when you execute shared_tools.py directly:
if vector_store is None:
    print("[skipped] needs sentence-transformers - see the cell above")
else:
    docs = get_resume_data.invoke({"query": "Skills"})

    print(f"Chunks returned: {len(docs)}\n")
    for i, doc in enumerate(docs, 1):
        print(f"--- chunk {i}  (page {doc.metadata.get('page')}) ---")
        print(doc.page_content)
        print()

[skipped] needs sentence-transformers - see the cell above


### 6c. Different query, different chunks

Change the topic and watch which parts of the resume come back. This is the whole
reason the tool's docstring says *"multiple calls with different topic queries are
usually needed"* — one search only ever sees a slice of the document.

In [13]:
if vector_store is None:
    print("[skipped] needs sentence-transformers")
else:
    for query in ["work experience", "education", "projects"]:
        docs = get_resume_data.invoke({"query": query})
        print(f"========== query: {query!r} -> {len(docs)} chunks ==========")
        print(docs[0].page_content[:300].strip())
        print()

[skipped] needs sentence-transformers


### 6d. What the return type costs you

`similarity_search` returns **`Document` objects**, not a string — even though the type
hint says `-> str`. The agent framework will stringify them, so the model ends up reading
Python `repr` output with `metadata=` noise in it.

Worth discussing: would the model do better if this tool returned clean joined text?

In [14]:
print("Declared return type :", get_resume_data.func.__annotations__.get("return"))

if vector_store is None:
    print("[skipped] needs sentence-transformers")
else:
    docs = get_resume_data.invoke({"query": "Skills"})
    print("Actual return type   :", type(docs).__name__, "of", type(docs[0]).__name__)
    print("\nWhat the model effectively reads:\n")
    print(str(docs)[:600], "...")

Declared return type : <class 'str'>
[skipped] needs sentence-transformers


---
## 7. The finished toolbox

All three tools in one list. That list is what `agent.py` hands to the LLM.

In [15]:
tools = [web_search_tool, get_job_recommendation, get_resume_data]

for t in tools:
    first_line = t.description.strip().split("\n")[0]
    print(f"{t.name:<25} args={list(t.args.keys())}")
    print(f"{'':<25} {first_line}")
    print()

duckduckgo_search         args=['query']
                          A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.

get_job_recommendation    args=['what', 'salary_min']
                          Searches live Indian job listings (Adzuna) by keyword and minimum salary.

get_resume_data           args=['query']
                          Retrieves chunks of the candidate's resume via similarity search.



### Recap — Part A

| Idea | Where you saw it |
|---|---|
| Pre-built tools exist — don't rewrite them | §2 `DuckDuckGoSearchRun` |
| `@tool` turns a function into an object; call it with `.invoke({...})` | §3 → §4 |
| The **docstring is the prompt** the model reads | §5 |
| Tools can wrap an API *or* a local vector store | §4 vs §6 |
| Return type shapes what the model sees | §6d |

Everything so far, *you* called by hand. From here the **model** decides which tool to
call, and when.

---
---
# Part B — `agent.py`: giving the tools to the model

A tool list on its own does nothing. `agent.py` wires three things together:

```
  LLM            (llm.py)          -> the brain that decides
+ tools          (shared_tools.py) -> the hands that act
+ system prompt  (system_prompts.py) -> the standing instructions
= agent
```

Then it wraps the whole thing in a Gradio web UI so a non-programmer can use it.

**Before running Part B you need Ollama running locally**, because `llm.py` uses
`ChatOllama`:

```bash
# once:
brew install ollama          # or download from https://ollama.com
ollama pull qwen3:0.6b
# every session:
ollama serve
```

The next cell checks for you. Cells that need the model are guarded, so you can read
straight through even if Ollama is not up yet.

In [16]:
# --- Is Ollama up? Everything in Part B depends on this. ---
import requests

OLLAMA_READY = False
MODEL_NAME = "qwen3:0.6b"   # must match the model in llm.py

try:
    r = requests.get("http://localhost:11434/api/tags", timeout=3)
    models = [m["name"] for m in r.json().get("models", [])]
    print("Ollama is running. Models available locally:")
    for m in models:
        print("  -", m)

    if any(m.startswith(MODEL_NAME.split(":")[0]) for m in models):
        OLLAMA_READY = True
    else:
        print(f"\n{MODEL_NAME} not pulled yet.  Run:  ollama pull {MODEL_NAME}")

except Exception as e:
    print("Ollama is NOT reachable on http://localhost:11434")
    print(f"  {type(e).__name__}: {e}\n")
    print("Start it with:  ollama serve        (and: ollama pull " + MODEL_NAME + ")")

print("\nOLLAMA_READY =", OLLAMA_READY)

Ollama is NOT reachable on http://localhost:11434
  ConnectionError: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/tags (Caused by NewConnectionError("HTTPConnection(host='localhost', port=11434): Failed to establish a new connection: [Errno 61] Connection refused"))

Start it with:  ollama serve        (and: ollama pull qwen3:0.6b)

OLLAMA_READY = False


---
## 8. The model — `llm.py`

```python
def create_llm():
    llm = ChatOllama(model="qwen3:0.6b", temperature=0.2)
    return llm
```

Two choices worth pausing on:

* **`qwen3:0.6b`** — a *tiny* model (0.6 billion parameters) so it runs on a laptop with
  no GPU. It is small enough that it will sometimes pick the wrong tool or pass silly
  arguments. That is not a bug in your code; it is the cost of the size. `llama3.2:3b`
  (the commented alternative) is noticeably better at tool calling if your machine can
  take it.
* **`temperature=0.2`** — low, i.e. near-deterministic. For an agent you want boring,
  repeatable decisions, not creative ones.

In [17]:
from llm import create_llm

llm = create_llm()
print("LLM object:", type(llm).__name__)
print("Model     :", llm.model)
print("Temp      :", llm.temperature)

LLM object: ChatOllama
Model     : qwen3:0.6b
Temp      : 0.2


In [18]:
# A plain call - no tools involved yet. This is just the model talking.
if not OLLAMA_READY:
    print("[skipped] start Ollama first - see the check cell above")
else:
    response = llm.invoke("In one sentence: what is a RAG pipeline?")
    print("Full response object type:", type(response).__name__)
    print("\n--- .content (the clean answer) ---")
    print(response.content)

[skipped] start Ollama first - see the check cell above


---
## 9. The system prompt — `system_prompts.py`

The system prompt is the agent's **standing orders**: it is prepended to every
conversation and the user never sees it.

Read `SYSTEM_PROMPT_LOCAL_TOOLS` below and notice how *unglamorous* it is. It is a
numbered procedure, not a personality. With a 0.6B model, vague instructions produce
vague behaviour, so every line does a specific job:

* **Step 1** tells it to fetch the resume *first* — otherwise it invents a candidate.
* **Step 2** does the thinking the model is bad at, in advance: how to map years →
  seniority, how many keywords to pick, what a realistic INR salary looks like.
* **Step 3** names the tool and its exact arguments.
* **Step 4 + Formatting Rules** pin down the output shape.
* *"Never list past companies from the user's resume as new openings"* — that line
  exists because the model did exactly that.

A system prompt is mostly a list of mistakes you have already seen.

In [19]:
from system_prompts import SYSTEM_PROMPT_LOCAL_TOOLS, SYSTEM_PROMPT_MCP_TOOLS

print(SYSTEM_PROMPT_LOCAL_TOOLS)

You are an Expert Job Matcher. Your task is to find live jobs matching the candidate's resume.

Follow these 4 execution steps sequentially:

1. RETRIEVE RESUME: Call `get_resume_data` using simple topic keywords (e.g., query="skills", query="experience").
2. EXTRACT LOGIC: From the retrieved resume:
   - Identify candidate experience level (Fresher: 0-1 yrs | Junior: 1-3 yrs | Mid: 3-5 yrs).
   - Pick 1 or 2 core tech keywords (e.g., "python", "fastapi").
   - Set a realistic minimum annual salary in INR (e.g., 300000).
3. SEARCH JOBS: Call `get_job_recommendation(what=..., salary_min=...)` using your extracted parameters.
4. FORMAT OUTPUT: Present top 3-5 live job results returned by the search tool.

Formatting Rules:
- Display: Job Title, Company, Location, Salary (in ₹ INR), Brief Description, and Apply Link (redirect_url).
- Base recommendations ONLY on live job API results. Never list past companies from the user's resume as new openings.
- If a field is missing in the result, w

There is a second prompt, `SYSTEM_PROMPT_MCP_TOOLS`, used in §11 when the tools move to
an MCP server. Compare the two — the difference is small but the reason matters.

In [20]:
import difflib

diff = difflib.unified_diff(
    SYSTEM_PROMPT_LOCAL_TOOLS.splitlines(),
    SYSTEM_PROMPT_MCP_TOOLS.splitlines(),
    fromfile="LOCAL_TOOLS", tofile="MCP_TOOLS", lineterm="",
)
print("\n".join(diff))

--- LOCAL_TOOLS
+++ MCP_TOOLS
@@ -1,16 +1,20 @@
 You are an Expert Job Matcher. Your task is to find live jobs matching the candidate's resume.
+
+Your tools are provided by a remote MCP server, so only rely on the tool names and
+descriptions given to you at runtime rather than assuming any tool exists beyond that.
 
 Follow these 4 execution steps sequentially:
 
-1. RETRIEVE RESUME: Call `get_resume_data` using simple topic keywords (e.g., query="skills", query="experience").
+1. RETRIEVE RESUME: Call the resume retrieval tool (`get_resume_data`) using simple topic keywords (e.g., query="skills", query="experience").
 2. EXTRACT LOGIC: From the retrieved resume:
    - Identify candidate experience level (Fresher: 0-1 yrs | Junior: 1-3 yrs | Mid: 3-5 yrs).
    - Pick 1 or 2 core tech keywords (e.g., "python", "fastapi").
    - Set a realistic minimum annual salary in INR (e.g., 300000).
-3. SEARCH JOBS: Call `get_job_recommendation(what=..., salary_min=...)` using your extracted para

---
## 10. Building the agent — `create_agent`

```python
job_search_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)
```

Three arguments, and you have an agent. What `create_agent` builds for you is a **loop**:

```
    user message
        |
        v
  +-> [ LLM decides ] --- no tool needed ---> final answer
  |         |
  |    wants a tool
  |         v
  |   [ run the tool ]
  |         |
  +--- feed result back
```

The LLM is called *repeatedly*. Each time it sees the conversation so far — including
the output of every tool it has already run — and decides whether to call another tool
or to answer. That loop is the entire difference between "an LLM" and "an agent".

In [21]:
from langchain.agents import create_agent

# Day-1 tool list: the three tools built in Part A, imported directly.
tools = [web_search_tool, get_job_recommendation, get_resume_data]

job_search_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT_LOCAL_TOOLS,
)

print("Agent built:", type(job_search_agent).__name__)
print("Tools bound:", [t.name for t in tools])

Agent built: CompiledStateGraph
Tools bound: ['duckduckgo_search', 'get_job_recommendation', 'get_resume_data']


### 10a. Running it — and reading the trace

`run_agent()` in `agent.py` prints the **whole response**, not just the answer:

```python
print("\n\n\nACTUAL RESPONSE WITH TOOLS\n\n\n", response)
return response["messages"][-1].content
```

That is deliberate. `response["messages"]` is the full transcript of the loop, and it is
the single best way to see what your agent actually did. You will see message types like:

| Message | Means |
|---|---|
| `HumanMessage` | what you asked |
| `AIMessage` with `tool_calls` | the model deciding to use a tool, and with what arguments |
| `ToolMessage` | what the tool returned |
| `AIMessage` with content | the final answer |

**When an agent misbehaves, read this trace before changing anything.** It tells you
whether the model picked the wrong tool, passed bad arguments, or got bad data back —
three completely different fixes.

In [22]:
if not OLLAMA_READY:
    print("[skipped] start Ollama first")
else:
    response = job_search_agent.invoke(
        {"messages": [{"role": "user", "content": "Find me the best jobs based on my resume data"}]}
    )

    print(f"Total messages in the loop: {len(response['messages'])}\n")

    for i, msg in enumerate(response["messages"], 1):
        kind = type(msg).__name__
        print(f"--- [{i}] {kind} ---")

        # An AIMessage that decided to call a tool:
        if getattr(msg, "tool_calls", None):
            for tc in msg.tool_calls:
                print(f"  CALLS TOOL: {tc['name']}")
                print(f"  WITH ARGS : {tc['args']}")
        elif kind == "ToolMessage":
            print(f"  from tool : {msg.name}")
            print(f"  returned  : {str(msg.content)[:300]}...")
        else:
            print(f"  {str(msg.content)[:400]}")
        print()

[skipped] start Ollama first


In [23]:
# The clean answer only - the last message in the list.
if not OLLAMA_READY:
    print("[skipped] start Ollama first")
else:
    print(response["messages"][-1].content)

[skipped] start Ollama first


---
## 11. Moving the tools out — `mcp_server.py`

So far the agent imports its tools straight from `shared_tools.py`. That works, but the
tools are welded to this one program. **MCP (Model Context Protocol)** lets you run the
tools as a separate service that any agent can connect to over HTTP.

`mcp_server.py` is the same two functions with a different decorator:

```python
mcp = FastMCP("Job Finding Tools Server")

@mcp.tool                          # instead of @tool
def get_job_recommendation(what: str, salary_min: int) -> dict:
    ...

mcp.run(transport="http", port=8001)
```

And `agent.py` fetches them at runtime instead of importing them:

```python
mcp_client = MultiServerMCPClient({
    "job_finding_tools": {"url": "http://localhost:8001/mcp", "transport": "streamable_http"}
})
mcp_tools = asyncio.run(mcp_client.get_tools())
tools = [web_search_tool, *mcp_tools]
```

**Why bother?**

| Imported tools | MCP tools |
|---|---|
| Must be in the same codebase / language | Any language, any machine |
| Restart the agent to change a tool | Restart only the server |
| One agent uses them | Any number of agents share them |
| Secrets live in the agent | Secrets live on the server |

Notice one thing in `mcp_server.py` that changed for a real reason: `get_resume_data`
returns `{"chunks": [...]}` — plain strings — instead of `Document` objects. MCP sends
JSON over the wire, so the tool has to return something serialisable. This also happens
to fix the messy output we saw back in §6d.

**To run this section, open a second terminal:**

```bash
source .venv/bin/activate
python mcp_server.py
```

> ### `asyncio.run()` vs `await` — read this before the next cell
>
> Fetching tools over the network is an **async** call. `agent.py` is a plain script, so
> it starts an event loop to run it:
>
> ```python
> mcp_tools = asyncio.run(mcp_client.get_tools())     # correct in a .py script
> ```
>
> A notebook **already has an event loop running**, so the same line raises
> `RuntimeError: asyncio.run() cannot be called from a running event loop`. In a
> notebook you `await` the coroutine directly instead:
>
> ```python
> mcp_tools = await mcp_client.get_tools()            # correct in a notebook
> ```
>
> Same call, two contexts. This trips up almost everyone the first time.

In [24]:
from langchain_mcp_adapters.client import MultiServerMCPClient

MCP_READY = False
mcp_tools = []

mcp_client = MultiServerMCPClient(
    {
        "job_finding_tools": {
            "url": "http://localhost:8001/mcp",
            "transport": "streamable_http",
        }
    }
)

try:
    # NOTE the `await` - see the box above. agent.py uses asyncio.run() instead,
    # because it is a script and not a notebook.
    mcp_tools = await mcp_client.get_tools()
    MCP_READY = True
    print(f"Connected. {len(mcp_tools)} tools fetched from the MCP server:\n")
    for t in mcp_tools:
        print(f"  {t.name:<25} args={list(t.args.keys())}")
except Exception as e:
    print("Could not fetch tools from http://localhost:8001/mcp")
    print(f"  {type(e).__name__}: {str(e)[:200]}\n")
    print("Start it in another terminal:  python mcp_server.py")

print("\nMCP_READY =", MCP_READY)

Could not fetch tools from http://localhost:8001/mcp
  ExceptionGroup: unhandled errors in a TaskGroup (1 sub-exception)

Start it in another terminal:  python mcp_server.py

MCP_READY = False


### 11a. The tools came across the network — but look identical

This is the payoff. The objects fetched over HTTP are the same *kind* of thing as the
`@tool`-decorated functions from Part A: same `.name`, same `.args`, same
`.description`. The agent cannot tell the difference, and neither can the LLM.

In [25]:
if not MCP_READY:
    print("[skipped] start mcp_server.py first")
else:
    remote_tool = next(t for t in mcp_tools if t.name == "get_job_recommendation")

    print("Type       :", type(remote_tool).__name__)
    print("Name       :", remote_tool.name)
    print("Args       :", remote_tool.args)
    print("\nDescription (came over HTTP from the server):")
    print(remote_tool.description[:400], "...")

[skipped] start mcp_server.py first


In [26]:
# Calling a remote tool looks exactly like calling a local one.
if not MCP_READY:
    print("[skipped] start mcp_server.py first")
else:
    resume_tool = next(t for t in mcp_tools if t.name == "get_resume_data")
    result = await resume_tool.ainvoke({"query": "skills"})   # await, not asyncio.run
    print("Returned plain JSON-safe data (not Document objects):\n")
    print(str(result)[:600])

[skipped] start mcp_server.py first


### 11b. The agent, rebuilt on MCP tools

Same `create_agent` call. Only the tool list changed — and the system prompt swaps to
`SYSTEM_PROMPT_MCP_TOOLS`, which avoids assuming a fixed tool set.

In [27]:
if not (OLLAMA_READY and MCP_READY):
    print("[skipped] needs BOTH Ollama and mcp_server.py running")
else:
    tools = [web_search_tool, *mcp_tools]

    job_search_agent = create_agent(
        model=llm,
        tools=tools,
        system_prompt=SYSTEM_PROMPT_MCP_TOOLS,
    )

    print("Agent rebuilt on MCP tools:", [t.name for t in tools])

[skipped] needs BOTH Ollama and mcp_server.py running


---
## 12. The UI — Gradio

Everything so far needed a Python prompt. `deploy_agent()` puts a web page in front of
it so anyone can use the agent.

```python
def deploy_agent():
    with gr.Blocks(title="Career Match Agent") as iface:
        gr.Markdown("## Career Match Agent")
        gr.Markdown("Finds job openings matched to your resume...")

        find_jobs_btn = gr.Button("Find Jobs", variant="primary")
        output = gr.Textbox(label="Recommended Jobs", lines=20)

        find_jobs_btn.click(fn=run_agent, inputs=None, outputs=output)

    iface.launch()
```

Read it as four steps:

1. **`gr.Blocks()`** — a container. Anything you create inside the `with` block is
   stacked onto the page, top to bottom.
2. **Components** — `gr.Markdown` (static text), `gr.Button` (input),
   `gr.Textbox` (output). These are just page elements; none of them do anything yet.
3. **`.click(fn=..., inputs=..., outputs=...)`** — the wiring, and the only line that
   matters. It says: *when this button is clicked, run `run_agent`, take nothing as
   input, and put whatever it returns into `output`.*
4. **`iface.launch()`** — starts a local web server on `http://127.0.0.1:7860`.

`inputs=None` because this agent takes no arguments — the resume is already in the
vector store. Add a `gr.Textbox` for a query and you would pass `inputs=that_textbox`,
and `run_agent` would need to accept a parameter.

In [28]:
import gradio as gr

def run_agent():
    """Exactly the function from agent.py. Gradio calls this on button click."""
    response = job_search_agent.invoke(
        {"messages": [{"role": "user", "content": "Find me the best jobs based on my resume data"}]}
    )
    return response["messages"][-1].content


def deploy_agent():
    with gr.Blocks(title="Career Match Agent") as iface:
        gr.Markdown("## Career Match Agent")
        gr.Markdown("Finds job openings matched to your resume, using live listings and your indexed resume data.")

        find_jobs_btn = gr.Button("Find Jobs", variant="primary")
        output = gr.Textbox(label="Recommended Jobs", lines=20)

        # The wiring: button -> function -> textbox
        find_jobs_btn.click(fn=run_agent, inputs=None, outputs=output)

    return iface


print("gradio version:", gr.__version__)
print("UI defined. Nothing is running yet - launch() does that.")

gradio version: 6.28.0
UI defined. Nothing is running yet - launch() does that.


### 12a. Launching it

`iface.launch()` **blocks** — it starts a server and keeps running until you stop it.
That is fine from a terminal, but in a notebook it will hang the cell, so the flag below
is off by default.

**Flip `LAUNCH_UI` to `True` and run the cell** to get the live app inline. Press the
stop button on the cell when you are done.

In [29]:
LAUNCH_UI = False   # <-- set to True to actually start the web app

if not LAUNCH_UI:
    print("LAUNCH_UI is False - set it to True and re-run this cell to start the app.")
elif not OLLAMA_READY:
    print("[skipped] start Ollama first, or the button will error when clicked")
else:
    iface = deploy_agent()
    iface.launch(inline=True)      # inline=True renders it inside the notebook

LAUNCH_UI is False - set it to True and re-run this cell to start the app.


> **Demo tip:** click *Find Jobs* and watch the terminal, not the browser. Because
> `run_agent` in `agent.py` prints the full response, you can see the agent's tool calls
> scroll past while the button spins. The UI shows the result; the terminal shows the
> reasoning.
>
> `iface.launch(share=True)` gives you a temporary public URL if you want to send it to
> someone — useful, but it exposes your local machine, so turn it off afterwards.

---
## Where everything lives

```
llm.py             the model                      -> §8
rag.py             PDF -> chunks -> vector store  -> §6
shared_tools.py    the three tools                -> §2-§7
system_prompts.py  standing orders                -> §9
mcp_server.py      tools as a remote service      -> §11
agent.py           model + tools + prompt + UI    -> §10, §12
```

### Recap — Part B

| Idea | Where you saw it |
|---|---|
| A small local model is cheap but fragile at tool-calling | §8 |
| A system prompt is a numbered procedure, not a personality | §9 |
| An agent is a **loop**: decide → call tool → read result → repeat | §10 |
| The message trace is your debugger | §10a |
| MCP turns tools into a shared service; the agent can't tell the difference | §11 |
| Gradio is three components and one `.click()` wiring line | §12 |

### Things to try

1. Swap `qwen3:0.6b` for `llama3.2:3b` in `llm.py` and re-run §10a. Count the tool calls
   in the trace — does the bigger model plan better?
2. Delete a rule from the system prompt (say, the "never list past companies" line) and
   see whether the failure it was guarding against comes back.
3. Add `web_search_tool` to `mcp_server.py` so all three tools come from MCP.
4. Give the Gradio app a text box so the user can type their own request, and change
   `run_agent(query)` to accept it.
5. From `rag.py`: `RetrievalQA` is deprecated. Rebuild that chain with LCEL.